In [ ]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold
from sklearn.utils import shuffle
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

In [4]:


def split_n_plus_1(X, y, n, random_state=42):
    X, y = shuffle(X, y, random_state=random_state)
    X_splits = np.array_split(X, n + 1)
    y_splits = np.array_split(y, n + 1)
    return X_splits, y_splits


In [5]:

def train_ensemble_svms(X_splits, y_splits, n):
    svms = []
    for i in range(n):
        svm = SVC(
            kernel="rbf",
            probability=True,   # bắt buộc
            C=1.0,
            gamma="scale"
        )
        svm.fit(X_splits[i], y_splits[i])
        svms.append(svm)
    return svms


In [6]:
def get_probability_features(models, X):
    """
    Output: (num_samples, n_models)
    """
    probs = []
    for model in models:
        p = model.predict_proba(X)[:, 1]  # lấy P(class=1)
        probs.append(p)

    return np.stack(probs, axis=1)


In [7]:
def train_meta_svm(Z, y):
    meta_svm = SVC(
        kernel="rbf",
        probability=True,
        C=1.0,
        gamma="scale"
    )
    meta_svm.fit(Z, y)
    return meta_svm


In [8]:
def predict_stack(models, meta_model, X):
    Z = get_probability_features(models, X)
    y_pred = meta_model.predict(Z)
    y_prob = meta_model.predict_proba(Z)[:, 1]
    return y_pred, y_prob


In [ ]:
# X: LFCC features (N, D)
# y: labels (0=real, 1=fake)

X1 = np.load("D:/Pythonfile/Voice-Activity-Detect/data/feature/train/mfcc_non-speech_1.npy")
y1 = np.zeros(len(X1), dtype=int)
X2 = np.load("D:/Pythonfile/Voice-Activity-Detect/data/feature/train/mfcc_non-speech_2.npy")
y2 = np.zeros(len(X2), dtype=int)
X3 = np.load("D:/Pythonfile/Voice-Activity-Detect/data/feature/train/mfcc_speech.npy")
y3 = np.ones(len(X3), dtype=int)

print(X1.shape, X2.shape, X3.shape)

X12 = np.concatenate((X1, X2), axis=0)
X = np.concatenate((X12, X3), axis=0)[:,:13]
y12 = np.concatenate((y1, y2), axis=0)
y = np.concatenate((y12, y3), axis=0)


n_folds = 5
skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)
accs, pres, recs, f1s, rocs = [], [], [], [], []

for fold_id, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    print(f"Fold {fold_id}: train={len(train_idx)}, test={len(test_idx)}")

    n = 5  # số SVM thành viên

    X_splits, y_splits = split_n_plus_1(X_train, y_train, n)
    ensemble_svms = train_ensemble_svms(X_splits, y_splits, n)

    X_meta = X_splits[n]      # (n+1)th dataset
    y_meta = y_splits[n]

    Z_meta = get_probability_features(ensemble_svms, X_meta)
    print(Z_meta.shape)
    final_svm = train_meta_svm(Z_meta, y_meta)
    y_pred, y_prob = predict_stack(ensemble_svms, final_svm, X_test)

    acc = accuracy_score(y_test, y_pred)
    pre = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1  = f1_score(y_test, y_pred)
    roc = roc_auc_score(y_test, y_prob)

    accs.append(acc)
    pres.append(pre)
    recs.append(rec)
    f1s.append(f1)
    rocs.append(roc)

    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {pre:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"ROC-AUC  : {roc:.4f}")


(10945435, 39) (1690403, 39) (12846082, 39)
Fold 0: train=20385536, test=5096384
